# FastConformer fine-tune - Kaggle chạy code từ GitHub

Notebook này dùng cho Kaggle GPU. Nó clone repo main từ GitHub, thêm `src/` vào `PYTHONPATH`, cài runtime NeMo, rồi chạy trực tiếp module `asr_lab.train.finetune_vivos`. Không dùng code-dataset, không dùng wrapper local.


## Cần bật trong Kaggle

- Accelerator: GPU, ưu tiên T4 hoặc P100.
- Internet: On, vì notebook cần `git clone`, tải package, tải model và tải VIVOS.
- Commit notebook sau khi chạy xong để Kaggle lưu output trong `/kaggle/working`.


## Phương pháp bản main

- Repo chạy: `https://github.com/ManhTanTran/ASR_local.git`.
- Model nền: `nvidia/stt_en_fastconformer_transducer_large`.
- Recipe: đổi vocabulary tiếng Việt, fine-tune full encoder + decoder RNNT trên VIVOS.
- Output chính: `/kaggle/working/runs/vivos-fc115m-v2norm/results.json` và `.nemo`.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import shlex
import shutil
import subprocess
import sys

GITHUB_REPO = "https://github.com/ManhTanTran/ASR_local.git"
GITHUB_REF = "main"
REPO_DIR = Path("/kaggle/working/ASR_local")

RUN_ID = "vivos-fc115m-v2norm"
PRETRAINED = "nvidia/stt_en_fastconformer_transducer_large"
EPOCHS = 50
BATCH = 16
VOCAB_SIZE = 1024
LR = "2e-4"
PRECISION = "32"
MAX_MINUTES = 480
CONSOLE_LOG_STEPS = 25

RUN_DIR = Path("/kaggle/working/runs") / RUN_ID
LOG_PATH = RUN_DIR / "train.log"

def cmd_text(cmd):
    return " ".join(shlex.quote(str(x)) for x in cmd)


def run(cmd, cwd=None, env=None, check=True):
    print("$", cmd_text(cmd), flush=True)
    return subprocess.run([str(x) for x in cmd], cwd=cwd, env=env, text=True, check=check)


def run_streamed(cmd, cwd=None, env=None, log_path=None, check=True):
    cmd = [str(x) for x in cmd]
    if log_path is None:
        return run(cmd, cwd=cwd, env=env, check=check)

    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    header = f"\n\n=== {datetime.now(timezone.utc).isoformat()} ===\n$ {cmd_text(cmd)}\n"
    print(header.strip(), flush=True)
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        log.write(header)
        proc = subprocess.Popen(
            cmd,
            cwd=cwd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
        rc = proc.wait()
        footer = f"\n=== exit code {rc} | log: {log_path} ===\n"
        log.write(footer)
    print(footer, flush=True)
    if check and rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)
    return subprocess.CompletedProcess(cmd, rc)


print("GitHub repo:", GITHUB_REPO)
print("GitHub ref :", GITHUB_REF)
print("Run ID     :", RUN_ID)
print("Train log  :", LOG_PATH)


## Clone code main từ GitHub


In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

run(["git", "clone", "--depth", "1", "--branch", GITHUB_REF, GITHUB_REPO, REPO_DIR])
run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"])

SRC_DIR = REPO_DIR / "src"
sys.path.insert(0, str(SRC_DIR))
print("SRC_DIR =", SRC_DIR)
print("has asr_lab:", (SRC_DIR / "asr_lab").is_dir())


## Cài runtime cho Kaggle GPU


In [ ]:
# Cài torch cu118 để chạy được cả P100 sm_60 và T4 trên Kaggle.
# Không import/reload torch trong notebook kernel; train sẽ chạy trong subprocess.
run([
    sys.executable, "-m", "pip", "-q", "install",
    "torch==2.7.1", "torchvision==0.22.1", "torchaudio==2.7.1",
    "--index-url", "https://download.pytorch.org/whl/cu118",
])
run([sys.executable, "-m", "pip", "-q", "install", "nemo_toolkit[asr]==2.7.3"])


## Verify nhanh GPU + NeMo trong subprocess


In [ ]:
verify_code = """
import torch
import nemo
print('torch', torch.__version__)
print('cuda ', torch.cuda.is_available())
print('nemo ', nemo.__version__)
if torch.cuda.is_available():
    layer = torch.nn.LSTM(8, 8).cuda()
    x = torch.randn(2, 3, 8).cuda()
    _ = layer(x)
    print('LSTM-cuda-OK')
"""
run([sys.executable, "-c", verify_code])


## Chạy fine-tune FastConformer theo main


In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(SRC_DIR) + os.pathsep + env.get("PYTHONPATH", "")
env["ASR_ARTIFACTS_DIR"] = "/kaggle/working"
env["ASR_DATA_DIR"] = "/tmp/vivos_data"
env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
env["PYTHONUNBUFFERED"] = "1"
env["ASR_CONSOLE_LOG_STEPS"] = str(CONSOLE_LOG_STEPS)

cmd = [
    sys.executable,
    "-u",
    "-m",
    "asr_lab.train.finetune_vivos",
    "--pretrained",
    PRETRAINED,
    "--run-id",
    RUN_ID,
    "--epochs",
    str(EPOCHS),
    "--batch",
    str(BATCH),
    "--vocab-size",
    str(VOCAB_SIZE),
    "--lr",
    LR,
    "--precision",
    PRECISION,
    "--max-minutes",
    str(MAX_MINUTES),
    "--console-log-steps",
    str(CONSOLE_LOG_STEPS),
]

print("Train log:", LOG_PATH)
run_streamed(cmd, cwd=REPO_DIR, env=env, log_path=LOG_PATH)


## Xem log train mới nhất

Cell train ở trên đã ghi toàn bộ stdout/stderr vào `train.log`. Nếu output Kaggle không tự cuộn, chạy cell này để xem đoạn cuối log.


In [ ]:
def show_tail(path, max_lines=220):
    path = Path(path)
    if not path.exists():
        print("Ch\u01b0a c\u00f3 log:", path)
        return
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("Log path:", path)
    print(f"Hi\u1ec3n th\u1ecb {min(len(lines), max_lines)}/{len(lines)} d\u00f2ng cu\u1ed1i")
    print("-" * 80)
    print("\n".join(lines[-max_lines:]))


show_tail(LOG_PATH)


## Đọc kết quả WER ngay trên Kaggle


In [ ]:
run_dir = Path("/kaggle/working/runs") / RUN_ID
results_path = run_dir / "results.json"
status_path = run_dir / "status.json"

print("run_dir:", run_dir)
print("status exists :", status_path.exists())
print("results exists:", results_path.exists())

if status_path.exists():
    print(json.dumps(json.loads(status_path.read_text()), ensure_ascii=False, indent=2))
if results_path.exists():
    results = json.loads(results_path.read_text())
    print(json.dumps(results, ensure_ascii=False, indent=2))


## Liệt kê artifact để tải về từ Kaggle Output


In [ ]:
for path in sorted(run_dir.rglob("*")):
    if path.is_file():
        size_mb = path.stat().st_size / 1_000_000
        print(f"{path.relative_to(run_dir)}\t{size_mb:.2f} MB")


## Sau khi chạy xong

Trong Kaggle, bấm `Save Version` hoặc `Commit` để lưu output. Artifact cần tải nằm trong phần Output của notebook, đặc biệt là thư mục `runs/vivos-fc115m-v2norm/` chứa `results.json`, `status.json`, `train.log`, `metrics.csv` và file `.nemo`.
